# pyebbr Quickstart: Empirical Bayes Binomial Estimation

This notebook demonstrates the core functionality of `pyebbr`, a Python port of the `ebbr` R package. We will use the classic example of baseball batting averages to show how empirical Bayes shrinkage improves estimates.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pyebbr import fit_beta_prior, add_ebb_estimate, ebb_prop_test

# Set seed for reproducibility
np.random.seed(42)

## 1. Load Data

We'll create a synthetic dataset representing baseball players with varying at-bats and hits.

In [ ]:
n_players = 100
at_bats = np.random.randint(10, 500, n_players)
true_average = 0.270
hits = np.random.binomial(at_bats, true_average)

df = pd.DataFrame({
    "player_id": range(n_players),
    "hits": hits,
    "at_bats": at_bats
})
df["raw_average"] = df["hits"] / df["at_bats"]
df.head()

## 2. Fit Beta Prior

We use `fit_beta_prior` to estimate the overall distribution of batting averages in the league.

In [ ]:
prior = fit_beta_prior(df["hits"], df["at_bats"])
print(f"Fitted Prior: Alpha={prior.alpha:.2f}, Beta={prior.beta:.2f}")
print(f"Prior Mean: {prior.mean:.3f}")

## 3. Shrink Estimates

Using `add_ebb_estimate`, we calculate the posterior (shrunken) mean and credible intervals for every player.

In [ ]:
shrunken_df = add_ebb_estimate(df, success_col="hits", total_col="at_bats", prior=prior)
shrunken_df.head()

## 4. Visualizing Shrinkage

Notice how players with fewer at-bats (small sample size) are pulled closer to the prior mean.

In [ ]:
plt.figure(figsize=(10, 6))
plt.scatter(shrunken_df["at_bats"], shrunken_df["raw_average"], alpha=0.5, label="Raw Average")
plt.scatter(shrunken_df["at_bats"], shrunken_df["ebb_fitted"], alpha=0.5, label="Shrunken (EBB)")
plt.axhline(prior.mean, color="red", linestyle="--", label="Prior Mean")
plt.xlabel("At Bats")
plt.ylabel("Batting Average")
plt.title("Empirical Bayes Shrinkage of Batting Averages")
plt.legend()
plt.show()

## 5. Bayesian Proportion Test

Let's test if a specific player's true average is likely above a threshold (e.g., 0.300).

In [ ]:
test_player = shrunken_df.iloc[0]
result = ebb_prop_test(successes=test_player["hits"], totals=test_player["at_bats"], prior=prior, threshold=0.300)

print(f"Player Hits: {test_player['hits']}, At Bats: {test_player['at_bats']}")
print(f"Probability true rate > 0.300: {result.prob_greater:.4f}")